<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "384"
iso_country = "Côte d'Ivoire"
year = 2025
import_portfolio = "h6_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_X.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 99.770*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Côte d'Ivoire': 

Number of trade records present in dataset: 34882
Number of partner countries present in dataset: 176
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$17179.5m
Total weight within the dataset: 23983.0 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,384,CÃ´te d'Ivoire,724,Spain,Import,Sea,690590,690590 | Ceramic chimney-pots,6905,"6905 | Roofing tiles, chimn",69,69 | Ceramic products,3735,"Non-refractory ceramic building bricks, floori...",Mining and Quarrying,34,NMM - Mineral products n.e.c.,9928.049,NaN,9928.049,kg,9317.0,9317.0,35730.055546,5013.0,3114.932823,1.112967e+05,['Container'],['Container'],367.776871,366.299595,366.404387,366.404387,367.369966,367.369966,386.652008,734.267478,428.937799,392.560123,394.142104,406.221724,377.704300,695.919777,445.206343,408.898360,399.647719,420.016625,369.863248,769.642455,558.759257,485.987251,449.193415,439.076749,362.127111,816.890142,652.277544,610.432387,477.471420,472.515914,354.53385,835.490585,782.381184,795.495002,516.853134,497.154491,346.796372,819.437939,791.593148,836.096469,623.969497,506.594101,339.439805,727.423285,743.389886,782.583487,587.225629,512.825459,332.044985,651.112554,668.827173,777.624422,637.023529,516.905280,3735
1,1,384,CÃ´te d'Ivoire,156,China,Import,Sea,690600,"690600 | Ceramic pipes, condu",6906,"6906 | Ceramic pipes, condu",69,69 | Ceramic products,3736,"Non-refractory ceramic pipes, conduits, gutter...",Mining and Quarrying,34,NMM - Mineral products n.e.c.,144592.436,NaN,144592.436,kg,4276.0,4276.0,145590.123808,20353.0,12646.763963,1.841244e+06,['Container'],['Container'],6084.338463,6059.899058,6061.632697,6061.632697,6077.606808,6077.606808,6396.600408,12147.397553,7096.157904,6494.341667,6520.513258,6720.353166,6248.573476,11512.990089,7365.297531,6764.634262,6611.595733,6948.569937,6118.854562,12732.625573,9243.866888,8039.958893,7431.257893,7263.892232,5990.871320,13514.270497,10790.992214,10098.724376,7899.076741,7817.095042,5865.25177,13821.988021,12943.369496,13160.318717,8550.590466,8224.704790,5737.246345,13556.420123,13095.768168,13832.011498,10322.676365,8380.869547,5615.542532,12034.170232,12298.314651,12946.716312,9714.801999,8483.958373,5493.205891,10771.719138,11064.781986,12864.675715,10538.636519,8551.453137,3736


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $3808.8m (22.2%), ... 

                               cifvalue    vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                               
10 | Cereals               1.448409e+09  3.207324e+09      8.4     13.4
03 | Fish and crustaceans  1.036235e+09  4.991921e+08      6.0      2.1
02 | Meat and edible meat  2.022937e+08  1.047866e+08      1.2      0.4
22 | Beverages, spirits a  1.570562e+08  2.007723e+08      0.9      0.8
23 | Food industries, res  1.535593e+08  2.717204e+08      0.9      1.1
19 | Preparations of cere  1.314329e+08  4.091079e+07      0.8      0.2
17 | Sugars and sugar con  1.150623e+08  2.304236e+08      0.7      1.0
04 | Dairy produce; birds  1.132126e+08  4.147637e+07      0.7      0.2
21 | Miscellaneous edible  9.371995e+07  2.438626e+07      0.5      0.1
11 | Products of the mill  9.289643e+07  1.909695e+08      0.5      0.8
07 | Vegetables and certa  7.48516

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $2765.2m (16.1%), ... 

                      cifvalue    vol_kg_vtw  pct_val  pct_vol
eora_desc                                                     
Agriculture       1.756419e+09  2.720975e+09     10.2     11.3
Food & Beverages  8.048164e+08  7.381165e+08      4.7      3.1
Fishing           2.039514e+08  9.825911e+07      1.2      0.4



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $2594.2m (15.1%), ... 

                                   cifvalue    vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                  
PCR - Processed rice           1.100316e+09  1.911250e+09      6.4      8.0
OFD - Food products n.e.c.     6.357910e+08  3.974867e+08      3.7      1.7
CMT - Bovine meat prods        1.989228e+08  1.032913e+08      1.2      0.4
OCR - Crops n.e.c.             1.511812e+08  3.201382e+07      0.9      0.1
VOL - Vegetable oils and fats  1.379240e+08  2.707837e+08      0.8      1.1
MIL - Dairy products           1.111516e+08  3.951267e+07      0.6      0.2
V_F - Vegetables, fruit, nuts  8.731899e+07  1.437257e+08      0.5      0.6
GRO - Cereal grains n.e.c.     6.658559e+07  2.754231e+08      0.4      1.1
SGR - Sugar                    6.272440e+07  1.377134e+08      0.4      0.6
OAP - Animal products n.e.c.   3.002943e+07  7.072671e+06      0.2      0.0
OMT - Meat products n

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 22.2% 
GTAP 15.1% 
EORA 16.1%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46363/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,384,CÃ´te d'Ivoire,724,Spain,Import,Sea,690590,690590 | Ceramic chimney-pots,6905,"6905 | Roofing tiles, chimn",69,69 | Ceramic products,3735,"Non-refractory ceramic building bricks, floori...",Mining and Quarrying,34,NMM - Mineral products n.e.c.,9928.049,NaN,9928.049,kg,9317.0,9317.0,35730.055546,5013.0,3114.932823,1.112967e+05,['Container'],['Container'],367.776871,366.299595,366.404387,366.404387,367.369966,367.369966,386.652008,734.267478,428.937799,392.560123,394.142104,406.221724,377.704300,695.919777,445.206343,408.898360,399.647719,420.016625,369.863248,769.642455,558.759257,485.987251,449.193415,439.076749,362.127111,816.890142,652.277544,610.432387,477.471420,472.515914,354.53385,835.490585,782.381184,795.495002,516.853134,497.154491,346.796372,819.437939,791.593148,836.096469,623.969497,506.594101,339.439805,727.423285,743.389886,782.583487,587.225629,512.825459,332.044985,651.112554,668.827173,777.624422,637.023529,516.905280,3735,NaN
1,1,384,CÃ´te d'Ivoire,156,China,Import,Sea,690600,"690600 | Ceramic pipes, condu",6906,"6906 | Ceramic pipes, condu",69,69 | Ceramic products,3736,"Non-refractory ceramic pipes, conduits, gutter...",Mining and Quarrying,34,NMM - Mineral products n.e.c.,144592.436,NaN,144592.436,kg,4276.0,4276.0,145590.123808,20353.0,12646.763963,1.841244e+06,['Container'],['Container'],6084.338463,6059.899058,6061.632697,6061.632697,6077.606808,6077.606808,6396.600408,12147.397553,7096.157904,6494.341667,6520.513258,6720.353166,6248.573476,11512.990089,7365.297531,6764.634262,6611.595733,6948.569937,6118.854562,12732.625573,9243.866888,8039.958893,7431.257893,7263.892232,5990.871320,13514.270497,10790.992214,10098.724376,7899.076741,7817.095042,5865.25177,13821.988021,12943.369496,13160.318717,8550.590466,8224.704790,5737.246345,13556.420123,13095.768168,13832.011498,10322.676365,8380.869547,5615.542532,12034.170232,12298.314651,12946.716312,9714.801999,8483.958373,5493.205891,10771.719138,11064.781986,12864.675715,10538.636519,8551.453137,3736,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $2058.6m of $17179.5m (12.0%)
Volume: 3798.6 of 23983.0 million kilograms (15.8%)
Value-to-Weight: 0.54 from 0.72 US$ per kg (+-24.3%)
Average distance to market (volume-weighted): 7432.1 from 5917.8 nm (+25.6%)

            food_group  N_com      cifvalue    vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    756  1.345511e+09  2.052130e+09      7.8      8.6
1  Unprocessed Cereals     36  3.476023e+08  1.295222e+09      2.0      5.4
2      Sugar and Honey    105  1.150740e+08  2.304270e+08      0.7      1.0
3                Dairy    155  1.104820e+08  4.045979e+07      0.6      0.2
4           Vegetables    456  9.780655e+07  1.455456e+08      0.6      0.6
5               Fruits    266  1.916999e+07  1.832633e+07      0.1      0.1
6        Fats and Oils    133  1.508809e+07  1.148700e+07      0.1      0.0
7        Meat and Prep     85  7.910183e+06  4.980067e+06      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,1.061030e+08,1.998280e+08,1.174487e+08,1.076306e+08,1.080572e+08,1.113950e+08,9.742641e+07,2.272905e+08,2.130700e+08,2.168269e+08,1.414441e+08,1.359756e+08,9.135131e+07,1.776618e+08,1.825015e+08,2.121335e+08,1.739116e+08,1.413078e+08
1,Fats and Oils,2.173203e+07,4.107238e+07,2.407646e+07,2.205397e+07,2.214215e+07,2.282260e+07,1.994291e+07,4.674914e+07,4.379013e+07,4.455201e+07,2.900582e+07,2.789052e+07,1.869019e+07,3.648864e+07,3.747693e+07,4.356268e+07,3.569637e+07,2.899137e+07
2,Fruits,2.902902e+07,5.398840e+07,3.200400e+07,2.944526e+07,2.955714e+07,3.044618e+07,2.670892e+07,6.168509e+07,5.768657e+07,5.881100e+07,3.854744e+07,3.703332e+07,2.508564e+07,4.852133e+07,4.974616e+07,5.765753e+07,4.713358e+07,3.848517e+07
3,Meat and Prep,8.324011e+06,1.580763e+07,9.234358e+06,8.451203e+06,8.485261e+06,8.745316e+06,7.632557e+06,1.798680e+07,1.684344e+07,1.712576e+07,1.112704e+07,1.070296e+07,7.148407e+06,1.401743e+07,1.439880e+07,1.674103e+07,1.371412e+07,1.112816e+07
4,Processed Cereals,6.888838e+09,1.258421e+10,7.556373e+09,6.965873e+09,6.992370e+09,7.211928e+09,6.358853e+09,1.446710e+10,1.359513e+10,1.386926e+10,9.180800e+09,8.709332e+09,5.987916e+09,1.138478e+10,1.166036e+10,1.349668e+10,1.113462e+10,9.029923e+09
5,Sugar and Honey,3.402351e+08,6.008671e+08,3.696421e+08,3.428735e+08,3.441120e+08,3.551323e+08,3.158115e+08,6.971293e+08,6.571095e+08,6.723495e+08,4.522335e+08,4.240053e+08,2.987243e+08,5.533336e+08,5.652873e+08,6.517523e+08,5.409739e+08,4.385491e+08
6,Unprocessed Cereals,2.013782e+09,3.744204e+09,2.220219e+09,2.040005e+09,2.047973e+09,2.111594e+09,1.853297e+09,4.284205e+09,4.019617e+09,4.094381e+09,2.687527e+09,2.565454e+09,1.740951e+09,3.356440e+09,3.442265e+09,3.992493e+09,3.283296e+09,2.663270e+09
7,Vegetables,2.444151e+08,4.595733e+08,2.703319e+08,2.479699e+08,2.489520e+08,2.565637e+08,2.244895e+08,5.240461e+08,4.905832e+08,4.995551e+08,3.257856e+08,3.130435e+08,2.105313e+08,4.097248e+08,4.205948e+08,4.884974e+08,4.001457e+08,3.253766e+08


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.85,0.10,0.01,0.02,0.05,1.18,1.05,1.08,0.40,0.35,0.78,0.83,1.09,0.75,0.45
1,Fats and Oils,1.28,0.16,0.02,0.03,0.07,1.78,1.58,1.63,0.60,0.53,1.18,1.25,1.65,1.13,0.68
2,Fruits,1.30,0.16,0.02,0.03,0.07,1.82,1.62,1.67,0.62,0.54,1.22,1.29,1.70,1.15,0.70
3,Meat and Prep,0.95,0.12,0.02,0.02,0.05,1.31,1.16,1.20,0.44,0.39,0.87,0.92,1.21,0.83,0.50
4,Processed Cereals,4.23,0.50,0.06,0.08,0.24,6.03,5.38,5.58,2.10,1.75,4.01,4.22,5.58,3.83,2.26
5,Sugar and Honey,2.26,0.26,0.02,0.03,0.13,3.31,2.97,3.10,1.19,0.94,2.21,2.32,3.07,2.11,1.22
6,Unprocessed Cereals,4.98,0.59,0.08,0.10,0.28,6.99,6.23,6.45,2.40,2.05,4.65,4.89,6.48,4.44,2.65
7,Vegetables,2.20,0.26,0.04,0.05,0.12,3.06,2.72,2.81,1.04,0.91,2.04,2.15,2.84,1.94,1.17
